In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever #capable. of doing semantic as well as keyword search

In [3]:
from pinecone import Pinecone, ServerlessSpec

indexName = "hybrid-search-langchain"
#initialise pinecone client
pc = Pinecone(api_key=api_key)

#create index
if indexName not in pc.list_indexes().names():
    pc.create_index(
        name = indexName,
        dimension = 384, #dimension of dense vector
        metric = "dotproduct",   #sparse value supported for dotproduct
        spec = ServerlessSpec(
            cloud = "aws",
            region = "us-east-1"
        ),   
    )

In [4]:
index = pc.Index(indexName)
index

In [ ]:
#vector embedding and sparse matrix
from langchain_huggingface import HuggingFaceEmbeddings

hf_api_token = os.getenv("HF_API_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
from pinecone_text.sparse import BM25Encoder     #uses tf-idf by default
sparseEncoder = BM25Encoder().default()
sparseEncoder

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/janayrawal/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
sentences = [
    "In 2023 I visited Paris.",
    "In 2022 I visited New York.",
    "In 2021 I visited New Orleans."
]

#tf-idf on these sentences
sparseEncoder.fit(sentences)

#store values to json file
sparseEncoder.dump("bm25_values.json")

#load to BM25Encoder object
sparseEncoder = BM25Encoder().load("bm25_values.json")

100%|██████████| 3/3 [00:00<00:00, 115.12it/s]


In [10]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=sparseEncoder,index=index)
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x30621cd90>, index=<pinecone.data.index.Index object at 0x1199f4ca0>)

In [11]:
retriever.add_texts(
    [
    "In 2023 I visited Paris.",
    "In 2022 I visited New York.",
    "In 2021 I visited New Orleans."
   ]
)

100%|██████████| 1/1 [00:03<00:00,  3.51s/it]


In [16]:
retriever.invoke("2021")

[Document(metadata={'score': 0.441410214}, page_content='In 2021 I visited New Orleans.'),
 Document(metadata={'score': 0.234355927}, page_content='In 2022 I visited New York.'),
 Document(metadata={'score': 0.222045898}, page_content='In 2023 I visited Paris.')]